# LeetCode 699 — Falling Squares bằng Python

- Link: https://leetcode.com/problems/falling-squares/
- Độ khó: **Hard**
- Chủ đề: Mô phỏng, Khoảng nửa mở `[l, r)`, Nén toạ độ (coordinate compression), Danh sách khoảng có thứ tự (`bisect`), Segment Tree với lazy propagation

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def fallingSquares(self, positions: List[List[int]]) -> List[int]:

```

## 1. Giới thiệu & mục tiêu học

Giống trò xếp gạch: các hình vuông lần lượt rơi xuống trục số, chồng lên nhau. Sau mỗi lần rơi, ta cần biết **chồng cao nhất** hiện tại.

Bài này là cửa ngõ rất tốt để học **Segment Tree** — cấu trúc trả lời "max trên một đoạn" và "gán giá trị cho cả đoạn" trong `O(log n)`. Nhưng ta sẽ đi từng bước: mô phỏng từng ô (dễ hiểu nhất), so từng cặp hình vuông (đủ qua LeetCode), danh sách khoảng có thứ tự, rồi mới tới Segment Tree.

Sau notebook này bạn sẽ:

1. Mô hình hoá đúng: hình vuông chiếm khoảng **nửa mở** `[left, left + side)` — chạm cạnh thì **không** đè lên nhau.
2. Viết mô phỏng từng ô làm trọng tài và lời giải `O(n²)` so từng cặp.
3. Duy trì "bản đồ độ cao" dạng danh sách khoảng rời nhau bằng `bisect`.
4. Nén toạ độ và cài Segment Tree "max trên đoạn + gán trên đoạn" có lazy propagation.
5. Kiểm chứng mọi cách bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Trên trục `X` có các hình vuông rơi lần lượt. `positions[i] = [left_i, sideLength_i]` nghĩa là hình vuông thứ `i` có cạnh `sideLength_i`, cạnh trái thẳng hàng với toạ độ `left_i`. Nó rơi thẳng xuống cho đến khi chạm **mặt trên của một hình vuông đã rơi** hoặc chạm **trục X**, rồi đứng yên.

Hình vuông chỉ dừng khi **đè lên phần có độ dài dương**; chỉ chạm cạnh bên của hình khác thì không tính (nó trượt xuống tiếp).

Trả về mảng `ans`, với `ans[i]` là **chiều cao của chồng cao nhất** sau khi hình vuông thứ `i` rơi xong.

**Ví dụ 1**

```text
Input:  positions = [[1,2],[2,3],[6,1]]
Output: [2,5,5]
Giải thích:
  Hình 1: [1,3), cao 2 -> chồng cao nhất 2.
  Hình 2: [2,5) đè lên hình 1 (phần [2,3)), đáy ở 2, đỉnh ở 2+3 = 5 -> 5.
  Hình 3: [6,7) không đè ai, cao 1 -> cao nhất vẫn 5.
```

**Ví dụ 2**

```text
Input:  positions = [[100,100],[200,100]]
Output: [100,100]
Giải thích: hình 2 [200,300) chỉ chạm cạnh hình 1 [100,200) tại x = 200 -> không đè lên, nằm trên trục X.
```

**Ràng buộc**

- `1 <= positions.length <= 1000`
- `1 <= left_i <= 10^8`
- `1 <= sideLength_i <= 10^6`

In [1]:
ex1 = [[1, 2], [2, 3], [6, 1]]
ex2 = [[100, 100], [200, 100]]

def draw(positions, width=None):
    """Vẽ chồng hình vuông bằng ký tự (chỉ dùng cho toạ độ nhỏ)."""
    width = width or max(l + s for l, s in positions) + 1
    h = [0] * width
    grid = {}
    for idx, (l, s) in enumerate(positions):
        base = max(h[l:l + s])
        for x in range(l, l + s):
            for y in range(base, base + s):
                grid[(x, y)] = str(idx + 1)
            h[x] = base + s
    top = max(h)
    for y in range(top - 1, -1, -1):
        print(f"{y + 1:>2} |" + ''.join(grid.get((x, y), '.') for x in range(width)))
    print("   +" + '-' * width)
    print("    " + ''.join(str(x % 10) for x in range(width)))

draw(ex1)

 5 |..222...
 4 |..222...
 3 |..222...
 2 |.11.....
 1 |.11...3.
   +--------
    01234567


## 3. Kiến thức nền

### 3.1. Khoảng nửa mở `[l, r)` và điều kiện giao nhau

Hình vuông `[left, side]` chiếm `[left, left + side)`. Hai khoảng nửa mở `[a, b)` và `[c, d)` **giao nhau với độ dài dương** khi và chỉ khi:

> `a < d` **và** `c < b`

Nếu `b == c` (chạm nhau tại một điểm) thì **không** giao.

In [2]:
def overlap(a, b, c, d):
    return a < d and c < b

print(overlap(1, 3, 2, 5))      # True: [1,3) và [2,5) chung [2,3)
print(overlap(100, 200, 200, 300))  # False: chỉ chạm tại 200
print(overlap(1, 5, 2, 3))      # True: chứa nhau

True
False
True


### 3.2. Max cộng dồn (prefix max)

Đáp án `ans[i]` là max của **mọi** đỉnh chồng tính tới thời điểm `i` — tức là "max cộng dồn" của dãy độ cao đỉnh các hình vuông.

In [3]:
from itertools import accumulate
tops = [2, 5, 1]
print(list(accumulate(tops, max)))

[2, 5, 5]


### 3.3. Nén toạ độ (coordinate compression)

Toạ độ tới `10^8` nhưng chỉ có ≤ `2n` điểm "quan trọng" (`left` và `left + side`). Sắp xếp các điểm đó; giữa hai điểm liên tiếp, độ cao **không đổi** → mỗi khoảng `[c_k, c_{k+1})` là một **ô** có chỉ số `k`. Khoảng `[l, r)` ứng với các ô `idx(l) .. idx(r) - 1`.

In [4]:
positions = ex1
coords = sorted({x for l, s in positions for x in (l, l + s)})
idx = {x: i for i, x in enumerate(coords)}
print("coords:", coords)
for l, s in positions:
    print(f"[{l},{l + s}) -> ô {idx[l]} .. {idx[l + s] - 1}")

coords: [1, 2, 3, 5, 6, 7]
[1,3) -> ô 0 .. 1
[2,5) -> ô 1 .. 2
[6,7) -> ô 4 .. 4


### 3.4. `bisect` trên danh sách mốc

`bisect_right(xs, x) - 1` cho chỉ số mốc lớn nhất `≤ x` — tức là khoảng `[xs[i], xs[i+1])` chứa `x`.

In [5]:
from bisect import bisect_right
xs = [0, 1, 3, 6]
for x in [0, 2, 3, 10]:
    print(x, "nằm trong khoảng bắt đầu ở", xs[bisect_right(xs, x) - 1])

0 nằm trong khoảng bắt đầu ở 0
2 nằm trong khoảng bắt đầu ở 1
3 nằm trong khoảng bắt đầu ở 3
10 nằm trong khoảng bắt đầu ở 6


### 3.5. Segment Tree — ý tưởng

Segment Tree là cây nhị phân mà mỗi nút quản lý một đoạn chỉ số `[lo, hi]` và lưu **max** trên đoạn đó. Truy vấn max một đoạn bất kỳ = ghép ≤ `O(log n)` nút. Để **gán cả đoạn** nhanh, ta dùng **lazy**: đánh dấu "cả đoạn này bằng v" ở nút, chỉ đẩy xuống con khi cần. Ví dụ nhỏ (mảng 4 phần tử) sẽ có ở Cách 4.

## 4. Ý tưởng nền tảng / trực giác

### 4.1. Một hình vuông rơi xuống dừng ở đâu?

Hình `[l, l + s)` rơi xuống và dừng ở độ cao `base` = **độ cao lớn nhất của mặt đất hiện tại trên khoảng `[l, l + s)`** (mặt đất = trục X + mặt trên các hình đã rơi). Đỉnh của nó ở `base + s`, và sau đó mặt đất trên cả khoảng `[l, l + s)` trở thành `base + s` (phẳng).

### 4.2. Tính tay ví dụ 1

| Hình | Khoảng | Mặt đất trên khoảng (trước) | `base` | Đỉnh | Max toàn cục |
|---|---|---|---|---|---|
| 1 | `[1,3)` | 0 | 0 | 2 | 2 |
| 2 | `[2,5)` | `[2,3)`: 2, `[3,5)`: 0 | 2 | 5 | 5 |
| 3 | `[6,7)` | 0 | 0 | 1 | 5 |

Đáp án `[2, 5, 5]`.

### 4.3. Hai thao tác cốt lõi

Mọi lời giải đều chỉ cần hai thao tác trên "bản đồ độ cao" `H(x)`:

1. **Truy vấn**: `max H(x)` với `x ∈ [l, r)`.
2. **Cập nhật**: gán `H(x) = v` với mọi `x ∈ [l, r)`.

Các cách khác nhau chỉ khác ở **cách lưu `H`**.

In [6]:
draw([[1, 5], [7, 1], [3, 5]])

10 |...33333.
 9 |...33333.
 8 |...33333.
 7 |...33333.
 6 |...33333.
 5 |.11111...
 4 |.11111...
 3 |.11111...
 2 |.11111...
 1 |.11111.2.
   +---------
    012345678


Ở hình trên: hình 3 `[3,8)` đè lên cả hình 1 (đỉnh 5) và hình 2 (đỉnh 1) → `base = 5` (lấy **max**), đỉnh 10.

## 5. Cách 1 — Brute Force: mô phỏng từng ô đơn vị

**Ý tưởng.** Lưu `H` bằng một mảng, mỗi phần tử là độ cao của ô `[x, x+1)`. Mỗi hình vuông: `base = max(H[l : l+s])`, gán `H[l : l+s] = base + s`. Chỉ dùng được khi toạ độ nhỏ — nhưng **cực kỳ dễ đúng**, là trọng tài hoàn hảo.

In [7]:
def falling_cells(positions):
    width = max(l + s for l, s in positions)
    H = [0] * (width + 1)
    ans, best = [], 0
    for l, s in positions:
        base = max(H[l:l + s])
        for x in range(l, l + s):
            H[x] = base + s
        best = max(best, base + s)
        ans.append(best)
    return ans

print(falling_cells(ex1))                    # [2, 5, 5]
print(falling_cells([[1, 5], [7, 1], [3, 5]]))

[2, 5, 5]
[5, 5, 10]


**Độ phức tạp.** `O(n · S)` thời gian với `S` là tổng độ dài cạnh, và `O(max toạ độ)` bộ nhớ. Với `left` tới `10^8` thì mảng không thể tạo nổi.

**Nút thắt.** Ta lưu độ cao **từng đơn vị** trong khi chỉ có ≤ `2n` điểm mà độ cao có thể thay đổi. Hướng cải tiến: chỉ làm việc với **các hình vuông** (Cách 2), hoặc **các khoảng** (Cách 3), hoặc **ô nén** (Cách 4).

## 6. Bước đột phá

**Insight 1 — chỉ cần các hình đã rơi.** Mặt đất trên `[l, r)` cao nhất bằng **đỉnh cao nhất trong các hình đã rơi có giao với `[l, r)`** (hoặc 0). Vậy:

> `top[i] = s_i + max{ top[j] : j < i, [l_j, r_j) giao [l_i, r_i) }` (max của tập rỗng = 0).

**Vì sao đúng?** Mỗi điểm `x` trong `[l, r)` có độ cao bằng đỉnh của hình **cuối cùng** phủ `x` — và đỉnh đó ≥ mọi hình phủ `x` trước nó (vì hình sau đè lên hình trước). Nên max trên `x ∈ [l, r)` của độ cao = max đỉnh của mọi hình có giao. → Cách 2, `O(n²)`.

**Insight 2 — mặt đất là hàm bậc thang.** `H(x)` là hằng số trên từng khoảng giữa các mốc. Lưu danh sách mốc đã sắp xếp + độ cao mỗi khoảng; mỗi lần rơi thì **cắt** tại `l` và `r`, lấy max các khoảng bên trong, rồi **gộp** chúng thành một khoảng. → Cách 3.

**Insight 3 — nén toạ độ + Segment Tree.** Sau khi nén, có ≤ `2n` ô. Hai thao tác "max trên đoạn" và "gán trên đoạn" đều `O(log n)` với Segment Tree lazy. → Cách 4, `O(n log n)`.

**Bất biến chung:** sau khi xử lý hình thứ `i`, cấu trúc dữ liệu mô tả đúng hàm mặt đất `H(x)` của `i` hình đầu, và `best` = max của `H`.

## 7. Các cách tiếp theo

Tổng cộng 4 cách:

| Cách | Lưu `H` thế nào | Time |
|---|---|---|
| 1 | Mảng từng ô đơn vị | `O(n · S)` |
| 2 | Không lưu `H`; so từng cặp hình | `O(n²)` |
| 3 | Danh sách khoảng có thứ tự (mốc + độ cao) | `O(n²)` tệ nhất, thường nhanh |
| 4 | Segment Tree trên ô nén | `O(n log n)` |

### 7.1. Cách 2 — So từng cặp hình vuông `O(n²)`

In [8]:
def falling_pairs(positions):
    tops = []                                   # tops[j] = đỉnh hình j
    ans, best = [], 0
    for i, (l, s) in enumerate(positions):
        r = l + s
        base = 0
        for j in range(i):
            lj, sj = positions[j]
            if l < lj + sj and lj < r:          # hai khoảng nửa mở giao nhau
                base = max(base, tops[j])
        tops.append(base + s)
        best = max(best, base + s)
        ans.append(best)
    return ans

print(falling_pairs(ex1), falling_pairs(ex2), falling_pairs([[1, 5], [7, 1], [3, 5]]))

[2, 5, 5] [100, 100] [5, 5, 10]


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def fallingSquares(self, positions: List[List[int]]) -> List[int]:
        tops, ans, best = [], [], 0
        for i, (l, s) in enumerate(positions):
            r, base = l + s, 0
            for j in range(i):
                lj, sj = positions[j]
                if l < lj + sj and lj < r:
                    base = max(base, tops[j])
            tops.append(base + s)
            best = max(best, base + s)
            ans.append(best)
        return ans
```

**Độ phức tạp.** `O(n²)` thời gian (≈ `5·10^5` phép với `n = 1000` — thoải mái), `O(n)` bộ nhớ.

### 7.2. Cách 3 — Danh sách khoảng có thứ tự (hàm bậc thang)

**Giải thích.** Lưu hai list song song:

- `xs`: các mốc tăng dần, bắt đầu `[0]`;
- `hs`: `hs[i]` là độ cao trên `[xs[i], xs[i+1])` (khoảng cuối kéo dài tới vô cực).

Thao tác `split(x)`: đảm bảo `x` là một mốc (nếu `x` nằm giữa khoảng `i` thì chèn mốc `x` với cùng độ cao `hs[i]`), trả về chỉ số của mốc `x`.

Khi rơi `[l, r)`: `i = split(l)`, `j = split(r)`; `base = max(hs[i:j])`; thay `xs[i:j]` bằng `[l]` và `hs[i:j]` bằng `[base + s]` (gộp mọi khoảng bên trong thành một).

In [9]:
from bisect import bisect_right

def falling_intervals(positions):
    xs, hs = [0], [0]

    def split(x):
        i = bisect_right(xs, x) - 1        # khoảng chứa x
        if xs[i] != x:
            xs.insert(i + 1, x)
            hs.insert(i + 1, hs[i])
            i += 1
        return i

    ans, best = [], 0
    for l, s in positions:
        r = l + s
        i = split(l)
        j = split(r)                       # r > l nên chèn r không làm lệch chỉ số i
        base = max(hs[i:j])
        xs[i:j] = [l]
        hs[i:j] = [base + s]
        best = max(best, base + s)
        ans.append(best)
    return ans

print(falling_intervals(ex1), falling_intervals(ex2), falling_intervals([[1, 5], [7, 1], [3, 5]]))

[2, 5, 5] [100, 100] [5, 5, 10]


Giải thích từng dòng:

- `bisect_right(xs, x) - 1`: mốc lớn nhất `≤ x` → khoảng chứa `x`.
- Nếu `x` chưa là mốc: chèn mốc mới ngay sau, độ cao bằng khoảng đang chứa nó (cắt đôi khoảng).
- `split(l)` trước rồi mới `split(r)`: vì `r > l`, mốc `r` được chèn **sau** `l` nên chỉ số `i` vẫn đúng.
- `xs[i:j] = [l]`: các mốc `l = xs[i] < xs[i+1] < … < xs[j-1]` bị thay bằng một mốc `l`; mốc `xs[j] = r` giữ nguyên → khoảng mới là `[l, r)`.

**Code nộp LeetCode (Cách 3):**

```python
from bisect import bisect_right

class Solution:
    def fallingSquares(self, positions: List[List[int]]) -> List[int]:
        xs, hs = [0], [0]
        def split(x):
            i = bisect_right(xs, x) - 1
            if xs[i] != x:
                xs.insert(i + 1, x)
                hs.insert(i + 1, hs[i])
                i += 1
            return i
        ans, best = [], 0
        for l, s in positions:
            i = split(l)
            j = split(l + s)
            base = max(hs[i:j])
            xs[i:j] = [l]
            hs[i:j] = [base + s]
            best = max(best, base + s)
            ans.append(best)
        return ans
```

**Độ phức tạp.** Mỗi lần rơi thêm tối đa 2 mốc và xoá các mốc bên trong → tổng số mốc bị duyệt trong `max(hs[i:j])` là `O(n)` **khấu hao**. Nhưng `list.insert`/gán lát cắt tốn `O(số mốc)` → `O(n²)` tệ nhất (thực tế rất nhanh vì chạy trong C). Bộ nhớ `O(n)`.

### 7.3. Cách 4 — Nén toạ độ + Segment Tree (lazy "gán đoạn", truy vấn max)

**Giải thích.**

1. Nén mọi `l` và `l + s` → `m` mốc → `m - 1` ô (ô `k` = `[coords[k], coords[k+1])`).
2. Segment Tree trên các ô, mỗi nút lưu `mx` (max đoạn) và `lazy` (giá trị gán chưa đẩy xuống; 0 = không có).
3. Mỗi hình: `base = query(idx[l], idx[r] - 1)`; `assign(idx[l], idx[r] - 1, base + s)`.

Ví dụ nhỏ để hiểu Segment Tree trước khi dùng:

In [10]:
class SegTree:
    """Segment Tree trên các ô 0..n-1: query max đoạn, assign (gán) đoạn, lazy propagation."""
    def __init__(self, n):
        self.n = n
        self.mx = [0] * (4 * n)
        self.lz = [0] * (4 * n)            # 0 = không có lệnh gán treo (độ cao luôn > 0 khi gán)

    def _apply(self, node, val):
        self.mx[node] = val
        self.lz[node] = val

    def _push(self, node):
        if self.lz[node]:
            self._apply(2 * node, self.lz[node])
            self._apply(2 * node + 1, self.lz[node])
            self.lz[node] = 0

    def query(self, ql, qr, node=1, lo=0, hi=None):
        if hi is None:
            hi = self.n - 1
        if qr < lo or hi < ql:
            return 0
        if ql <= lo and hi <= qr:
            return self.mx[node]
        self._push(node)
        mid = (lo + hi) // 2
        return max(self.query(ql, qr, 2 * node, lo, mid),
                   self.query(ql, qr, 2 * node + 1, mid + 1, hi))

    def assign(self, ql, qr, val, node=1, lo=0, hi=None):
        if hi is None:
            hi = self.n - 1
        if qr < lo or hi < ql:
            return
        if ql <= lo and hi <= qr:
            self._apply(node, val)
            return
        self._push(node)
        mid = (lo + hi) // 2
        self.assign(ql, qr, val, 2 * node, lo, mid)
        self.assign(ql, qr, val, 2 * node + 1, mid + 1, hi)
        self.mx[node] = max(self.mx[2 * node], self.mx[2 * node + 1])

t = SegTree(4)
t.assign(1, 2, 5)
print("max[0..3] =", t.query(0, 3), "| max[0..0] =", t.query(0, 0), "| max[2..3] =", t.query(2, 3))
t.assign(0, 1, 7)
print("sau gán [0..1]=7: max[1..2] =", t.query(1, 2), "| max[2..2] =", t.query(2, 2))

max[0..3] = 5 | max[0..0] = 0 | max[2..3] = 5
sau gán [0..1]=7: max[1..2] = 7 | max[2..2] = 5


Giải thích Segment Tree:

- Nút `1` quản lý `[0, n-1]`; nút `node` có con `2·node` (nửa trái) và `2·node+1` (nửa phải). Mảng `4n` là đủ chỗ.
- `_apply(node, val)`: gán cả đoạn của nút = `val` → `mx = val`, ghi `lz = val` để nhớ "con của tôi cũng phải bằng `val`".
- `_push(node)`: trước khi đi xuống con, đẩy lệnh gán treo xuống hai con.
- `query`: đoạn nút nằm ngoài → 0; nằm trọn trong → trả `mx`; còn lại chia đôi.
- `assign`: tương tự, sau khi gán hai con thì cập nhật `mx` của nút từ hai con.

Giờ ghép vào bài:

In [11]:
def falling_segtree(positions):
    coords = sorted({x for l, s in positions for x in (l, l + s)})
    idx = {x: i for i, x in enumerate(coords)}
    tree = SegTree(len(coords))             # ô k = [coords[k], coords[k+1]); ô cuối không bao giờ dùng
    ans, best = [], 0
    for l, s in positions:
        a, b = idx[l], idx[l + s] - 1        # các ô bị hình vuông phủ
        base = tree.query(a, b)
        tree.assign(a, b, base + s)
        best = max(best, base + s)
        ans.append(best)
    return ans

print(falling_segtree(ex1), falling_segtree(ex2), falling_segtree([[1, 5], [7, 1], [3, 5]]))

[2, 5, 5] [100, 100] [5, 5, 10]


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def fallingSquares(self, positions: List[List[int]]) -> List[int]:
        coords = sorted({x for l, s in positions for x in (l, l + s)})
        idx = {x: i for i, x in enumerate(coords)}
        n = len(coords)
        mx, lz = [0] * (4 * n), [0] * (4 * n)

        def push(node):
            if lz[node]:
                for c in (2 * node, 2 * node + 1):
                    mx[c] = lz[c] = lz[node]
                lz[node] = 0

        def query(node, lo, hi, ql, qr):
            if qr < lo or hi < ql: return 0
            if ql <= lo and hi <= qr: return mx[node]
            push(node)
            mid = (lo + hi) // 2
            return max(query(2*node, lo, mid, ql, qr), query(2*node+1, mid+1, hi, ql, qr))

        def assign(node, lo, hi, ql, qr, val):
            if qr < lo or hi < ql: return
            if ql <= lo and hi <= qr:
                mx[node] = lz[node] = val
                return
            push(node)
            mid = (lo + hi) // 2
            assign(2*node, lo, mid, ql, qr, val)
            assign(2*node+1, mid+1, hi, ql, qr, val)
            mx[node] = max(mx[2*node], mx[2*node+1])

        ans, best = [], 0
        for l, s in positions:
            a, b = idx[l], idx[l + s] - 1
            base = query(1, 0, n - 1, a, b)
            assign(1, 0, n - 1, a, b, base + s)
            best = max(best, base + s)
            ans.append(best)
        return ans
```

**Độ phức tạp.** Nén `O(n log n)`; mỗi truy vấn/gán `O(log n)` → tổng `O(n log n)`, bộ nhớ `O(n)`.

## 8. TRACE từng bước

### 8.1. Bảng trạng thái Cách 2 cho `[[1,5],[7,1],[3,5]]`

| i | Khoảng | So với hình j | Giao? | `tops[j]` | `base` | Đỉnh | `best` |
|---|---|---|---|---|---|---|---|
| 0 | `[1,6)` | — | — | — | 0 | 5 | 5 |
| 1 | `[7,8)` | j=0 `[1,6)` | không | 5 | 0 | 1 | 5 |
| 2 | `[3,8)` | j=0 `[1,6)` | **có** | 5 | 5 | | |
|   |         | j=1 `[7,8)` | **có** | 1 | max(5,1)=5 | 10 | 10 |

Đáp án `[5, 5, 10]`.

### 8.2. `trace_pairs`

In [12]:
def trace_pairs(positions):
    tops, best = [], 0
    for i, (l, s) in enumerate(positions):
        r, base = l + s, 0
        hits = []
        for j in range(i):
            lj, sj = positions[j]
            if l < lj + sj and lj < r:
                hits.append(f"#{j}[{lj},{lj + sj}) đỉnh {tops[j]}")
                base = max(base, tops[j])
        tops.append(base + s)
        best = max(best, base + s)
        print(f"hình #{i} [{l},{r}): giao {hits if hits else 'không ai'} -> base={base}, đỉnh={base + s}, best={best}")

trace_pairs([[1, 5], [7, 1], [3, 5]])

hình #0 [1,6): giao không ai -> base=0, đỉnh=5, best=5
hình #1 [7,8): giao không ai -> base=0, đỉnh=1, best=5
hình #2 [3,8): giao ['#0[1,6) đỉnh 5', '#1[7,8) đỉnh 1'] -> base=5, đỉnh=10, best=10


### 8.3. `trace_intervals` — hàm bậc thang thay đổi thế nào

In [13]:
def trace_intervals(positions):
    xs, hs = [0], [0]
    def split(x):
        i = bisect_right(xs, x) - 1
        if xs[i] != x:
            xs.insert(i + 1, x)
            hs.insert(i + 1, hs[i])
            i += 1
        return i
    def fmt():
        return ' '.join(f"[{xs[k]},{xs[k + 1] if k + 1 < len(xs) else '∞'}):{hs[k]}" for k in range(len(xs)))
    print("ban đầu:", fmt())
    best = 0
    for l, s in positions:
        i = split(l)
        j = split(l + s)
        print(f"rơi [{l},{l + s}) — sau khi cắt: {fmt()}  | các khoảng bên trong: hs[{i}:{j}] = {hs[i:j]}")
        base = max(hs[i:j])
        xs[i:j] = [l]
        hs[i:j] = [base + s]
        best = max(best, base + s)
        print(f"   base={base}, gộp thành [{l},{l + s}):{base + s} -> {fmt()}   best={best}")

trace_intervals([[1, 5], [7, 1], [3, 5]])

ban đầu: [0,∞):0
rơi [1,6) — sau khi cắt: [0,1):0 [1,6):0 [6,∞):0  | các khoảng bên trong: hs[1:2] = [0]
   base=0, gộp thành [1,6):5 -> [0,1):0 [1,6):5 [6,∞):0   best=5
rơi [7,8) — sau khi cắt: [0,1):0 [1,6):5 [6,7):0 [7,8):0 [8,∞):0  | các khoảng bên trong: hs[3:4] = [0]
   base=0, gộp thành [7,8):1 -> [0,1):0 [1,6):5 [6,7):0 [7,8):1 [8,∞):0   best=5
rơi [3,8) — sau khi cắt: [0,1):0 [1,3):5 [3,6):5 [6,7):0 [7,8):1 [8,∞):0  | các khoảng bên trong: hs[2:5] = [5, 0, 1]
   base=5, gộp thành [3,8):10 -> [0,1):0 [1,3):5 [3,8):10 [8,∞):0   best=10


### 8.4. `trace_segtree` — độ cao các ô nén sau mỗi lần rơi

In [14]:
def trace_segtree(positions):
    coords = sorted({x for l, s in positions for x in (l, l + s)})
    idx = {x: i for i, x in enumerate(coords)}
    tree = SegTree(len(coords))
    cells = [f"[{coords[k]},{coords[k + 1]})" for k in range(len(coords) - 1)]
    print("coords:", coords)
    print("ô     :", cells)
    best = 0
    for l, s in positions:
        a, b = idx[l], idx[l + s] - 1
        base = tree.query(a, b)
        tree.assign(a, b, base + s)
        best = max(best, base + s)
        heights = [tree.query(k, k) for k in range(len(coords) - 1)]
        print(f"rơi [{l},{l + s}) -> ô {a}..{b}, base={base}, đỉnh={base + s}; độ cao các ô = {heights}; best={best}")

trace_segtree([[1, 5], [7, 1], [3, 5]])
print()
trace_segtree(ex1)

coords: [1, 3, 6, 7, 8]
ô     : ['[1,3)', '[3,6)', '[6,7)', '[7,8)']
rơi [1,6) -> ô 0..1, base=0, đỉnh=5; độ cao các ô = [5, 5, 0, 0]; best=5
rơi [7,8) -> ô 3..3, base=0, đỉnh=1; độ cao các ô = [5, 5, 0, 1]; best=5
rơi [3,8) -> ô 1..3, base=5, đỉnh=10; độ cao các ô = [5, 10, 10, 10]; best=10

coords: [1, 2, 3, 5, 6, 7]
ô     : ['[1,2)', '[2,3)', '[3,5)', '[5,6)', '[6,7)']
rơi [1,3) -> ô 0..1, base=0, đỉnh=2; độ cao các ô = [2, 2, 0, 0, 0]; best=2
rơi [2,5) -> ô 1..2, base=2, đỉnh=5; độ cao các ô = [2, 5, 5, 0, 0]; best=5
rơi [6,7) -> ô 4..4, base=0, đỉnh=1; độ cao các ô = [2, 5, 5, 0, 1]; best=5


## 9. Code nộp LeetCode

Chọn **Cách 4** (nén toạ độ + Segment Tree) — `O(n log n)`, là kỹ thuật tổng quát nhất. (Cách 2 `O(n²)` cũng được chấp nhận với `n ≤ 1000`.)

In [15]:
from typing import List

class Solution:
    def fallingSquares(self, positions: List[List[int]]) -> List[int]:
        coords = sorted({x for l, s in positions for x in (l, l + s)})
        idx = {x: i for i, x in enumerate(coords)}
        n = len(coords)
        mx, lz = [0] * (4 * n), [0] * (4 * n)

        def push(node):
            if lz[node]:
                for c in (2 * node, 2 * node + 1):
                    mx[c] = lz[c] = lz[node]
                lz[node] = 0

        def query(node, lo, hi, ql, qr):
            if qr < lo or hi < ql:
                return 0
            if ql <= lo and hi <= qr:
                return mx[node]
            push(node)
            mid = (lo + hi) // 2
            return max(query(2 * node, lo, mid, ql, qr), query(2 * node + 1, mid + 1, hi, ql, qr))

        def assign(node, lo, hi, ql, qr, val):
            if qr < lo or hi < ql:
                return
            if ql <= lo and hi <= qr:
                mx[node] = lz[node] = val
                return
            push(node)
            mid = (lo + hi) // 2
            assign(2 * node, lo, mid, ql, qr, val)
            assign(2 * node + 1, mid + 1, hi, ql, qr, val)
            mx[node] = max(mx[2 * node], mx[2 * node + 1])

        ans, best = [], 0
        for l, s in positions:
            a, b = idx[l], idx[l + s] - 1
            base = query(1, 0, n - 1, a, b)
            assign(1, 0, n - 1, a, b, base + s)
            best = max(best, base + s)
            ans.append(best)
        return ans

assert Solution().fallingSquares([[1, 2], [2, 3], [6, 1]]) == [2, 5, 5]
assert Solution().fallingSquares([[100, 100], [200, 100]]) == [100, 100]
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Mô hình.** Gọi `H_i(x)` là độ cao mặt đất tại `x` sau `i` hình. Hình `i+1` chiếm `[l, r)` sẽ dừng tại `base = max_{x ∈ [l, r)} H_i(x)` (nó rơi tới khi chạm điểm cao nhất bên dưới với phần giao độ dài dương; vì `H_i` là hàm bậc thang hằng trên các khoảng nửa mở nên "max" luôn đạt trên một đoạn có độ dài dương). Sau đó `H_{i+1}(x) = base + s` với `x ∈ [l, r)` và không đổi ở nơi khác. `ans[i]` = `max_x H_{i}(x)` = max các đỉnh đã có.

**Cách 2.** `H_i(x)` = đỉnh của hình cuối cùng phủ `x` (hoặc 0). Các hình phủ `x` có đỉnh **tăng dần** theo thời gian (hình sau đè lên hình trước nên đỉnh cao hơn), nên `H_i(x) = max đỉnh các hình phủ x`. Lấy max trên `x ∈ [l, r)` = max đỉnh các hình có giao với `[l, r)` → đúng công thức Cách 2.

**Cách 3.** Bất biến: `(xs, hs)` mô tả đúng `H_i`. `split` không đổi hàm (chỉ cắt một khoảng thành hai khoảng cùng độ cao). Sau khi cắt, các khoảng `i..j-1` phủ đúng `[l, r)` → `max(hs[i:j])` là `base`; thay bằng một khoảng `[l, r)` cao `base + s` chính là cập nhật `H_{i+1}`.

**Cách 4.** Sau nén, `H_i` hằng số trên mỗi ô `[coords[k], coords[k+1])` (vì mọi điểm gãy đều là một `l` hoặc `r`). Khoảng `[l, r)` = hợp đúng các ô `idx[l] .. idx[r] - 1`. Segment Tree trả lời đúng max và gán đúng (lazy chỉ trì hoãn, không làm sai giá trị khi đọc vì luôn `push` trước khi đi xuống).

## 11. Phân tích Time/Space

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Từng ô | `O(n · S)` | `O(max toạ độ)` | chỉ cho toạ độ nhỏ |
| 2. Từng cặp | `O(n²)` | `O(n)` | đủ cho `n ≤ 1000` |
| 3. Danh sách khoảng | `O(n²)` tệ nhất (insert), khấu hao tốt | `O(n)` | rất nhanh thực tế |
| 4. Segment Tree | `O(n log n)` | `O(n)` | tổng quát, mở rộng được |

Đo thời gian với `n = 1000` hình, toạ độ lớn:

In [16]:
import random, time

random.seed(699)
big = [[random.randint(1, 10**8 // 1000), random.randint(1, 10**6 // 100)] for _ in range(1000)]
res = []
for name, fn in [("pairs", falling_pairs), ("intervals", falling_intervals), ("segtree", falling_segtree)]:
    t0 = time.perf_counter()
    res.append(fn(big))
    print(f"{name:<10}: max cuối = {res[-1][-1]:>8}  {time.perf_counter() - t0:.3f}s")
assert res[0] == res[1] == res[2]

pairs     : max cuối =   844853  0.026s
intervals : max cuối =   844853  0.001s
segtree   : max cuối =   844853  0.012s


## 12. Bộ test

1. Ví dụ đề.
2. Edge case: một hình; các hình chạm cạnh nhau (không chồng); hình giống hệt nhau chồng lên; hình to phủ nhiều hình nhỏ; hình nhỏ rơi lên giữa hình to.
3. Ngẫu nhiên có seed với toạ độ nhỏ: mọi cách so với mô phỏng từng ô.

In [17]:
METHODS = [falling_pairs, falling_intervals, falling_segtree, lambda p: Solution().fallingSquares(p)]

def check(positions, expected=None):
    ref = falling_cells(positions)
    for f in METHODS:
        got = f([p[:] for p in positions])
        assert got == ref, (positions, got, ref)
    if expected is not None:
        assert ref == expected, (positions, ref, expected)
    return ref

check([[1, 2], [2, 3], [6, 1]], [2, 5, 5])
check([[100, 100], [200, 100]], [100, 100])
check([[5, 3]], [3])
check([[1, 2], [3, 2], [5, 2]], [2, 2, 2])                 # chạm cạnh liên tiếp
check([[2, 3], [2, 3], [2, 3]], [3, 6, 9])                 # chồng thẳng
check([[1, 1], [3, 1], [5, 1], [1, 6]], [1, 1, 1, 7])      # hình to phủ nhiều hình nhỏ
check([[1, 10], [5, 1], [3, 2]], [10, 11, 12])             # hình nhỏ trên hình to
check([[1, 5], [7, 1], [3, 5]], [5, 5, 10])
check([[4, 1], [3, 1], [5, 1], [3, 3]], [1, 1, 1, 4])
print("Ví dụ + edge case: OK")

random.seed(2028)
for _ in range(2000):
    n = random.randint(1, 8)
    pos = [[random.randint(1, 12), random.randint(1, 4)] for _ in range(n)]
    check(pos)
print("2000 bộ ngẫu nhiên: mọi cách khớp mô phỏng từng ô")

Ví dụ + edge case: OK
2000 bộ ngẫu nhiên: mọi cách khớp mô phỏng từng ô


## 13. Lỗi người mới hay gặp

### Lỗi 1: Coi chạm cạnh là đè lên (dùng khoảng đóng)

In [18]:
def falling_closed(positions):
    tops, ans, best = [], [], 0
    for i, (l, s) in enumerate(positions):
        base = 0
        for j in range(i):
            lj, sj = positions[j]
            if l <= lj + sj and lj <= l + s:          # SAI: <= coi chạm cạnh là giao
                base = max(base, tops[j])
        tops.append(base + s)
        best = max(best, base + s)
        ans.append(best)
    return ans

print("sai:", falling_closed(ex2), "| đúng:", falling_cells(ex2))

sai: [100, 200] | đúng: [100, 100]


**Vì sao sai:** hình `[100,200)` và `[200,300)` chỉ chạm tại `x = 200`, độ dài giao bằng 0 → không đè. Điều kiện đúng với khoảng nửa mở: `l < lj + sj and lj < l + s`.

### Lỗi 2: Trả về đỉnh của hình hiện tại thay vì **max toàn cục**

In [19]:
def falling_no_prefix_max(positions):
    tops = []
    for i, (l, s) in enumerate(positions):
        base = max([tops[j] for j in range(i)
                    if l < positions[j][0] + positions[j][1] and positions[j][0] < l + s] + [0])
        tops.append(base + s)
    return tops                                       # SAI: thiếu max cộng dồn

print("sai:", falling_no_prefix_max(ex1), "| đúng:", falling_cells(ex1))

sai: [2, 5, 1] | đúng: [2, 5, 5]


**Vì sao sai:** đề hỏi chồng **cao nhất** trên toàn trục sau mỗi lần rơi. Hình 3 thấp (1) nhưng chồng cao nhất vẫn là 5. Cần `best = max(best, top)`.

### Lỗi 3: Lấy đỉnh của hình giao **gần nhất theo thời gian** thay vì **max**

In [20]:
def falling_last_hit(positions):
    tops, ans, best = [], [], 0
    for i, (l, s) in enumerate(positions):
        base = 0
        for j in range(i):
            lj, sj = positions[j]
            if l < lj + sj and lj < l + s:
                base = tops[j]                        # SAI: ghi đè thay vì lấy max
        tops.append(base + s)
        best = max(best, base + s)
        ans.append(best)
    return ans

p = [[1, 5], [7, 1], [3, 5]]
print("sai:", falling_last_hit(p), "| đúng:", falling_cells(p))

sai: [5, 5, 6] | đúng: [5, 5, 10]


**Vì sao sai:** hình mới phủ một **khoảng**, các hình bên dưới có thể nằm ở những vị trí khác nhau; hình rơi dừng ở **điểm cao nhất** bên dưới. Phải `base = max(base, tops[j])`.

### Lỗi 4: Nén toạ độ nhưng phủ cả ô `idx[r]`

In [21]:
def falling_seg_inclusive(positions):
    coords = sorted({x for l, s in positions for x in (l, l + s)})
    idx = {x: i for i, x in enumerate(coords)}
    tree = SegTree(len(coords))
    ans, best = [], 0
    for l, s in positions:
        a, b = idx[l], idx[l + s]                     # SAI: phải là idx[l + s] - 1
        base = tree.query(a, b)
        tree.assign(a, b, base + s)
        best = max(best, base + s)
        ans.append(best)
    return ans

print("sai:", falling_seg_inclusive(ex2), "| đúng:", falling_segtree(ex2))

sai: [100, 200] | đúng: [100, 100]


**Vì sao sai:** ô `idx[r]` là `[r, …)` — nằm **ngoài** hình vuông. Phủ thêm ô đó làm hình chạm cạnh bị coi là đè lên. Đây chính là Lỗi 1 "đội lốt" nén toạ độ.

## 14. Mẹo & nhận dạng pattern

- **"Cập nhật cả đoạn + hỏi max/min/tổng trên đoạn"** → Segment Tree có lazy (hoặc danh sách khoảng nếu thao tác là "gán").
- **Toạ độ lớn (`10^8`, `10^9`) nhưng ít sự kiện (`≤ 10^4`)** → nén toạ độ, hoặc Segment Tree động.
- **"Gán cả đoạn = một giá trị"** → cấu trúc "danh sách khoảng rời nhau" (còn gọi Chtholly Tree / ODT) rất gọn, giống *Range Module* (LeetCode 715) và *My Calendar III* (732).
- **Khoảng nửa mở `[l, r)`** giúp tránh lỗi chạm cạnh và làm độ dài = `r - l`.
- Khi `n` nhỏ (≤ 1000), lời giải `O(n²)` so từng cặp thường đủ — hãy viết nó trước như một baseline/đối chiếu.

## 15. Biến thể

1. **Trả về độ cao chồng tại một điểm truy vấn `x`** → `query(idx(x), idx(x))` (cần thêm `x` vào tập toạ độ nén).
2. **Hình chữ nhật (rộng `w`, cao `h`)** thay cho hình vuông → đỉnh = `base + h`, khoảng `[l, l + w)`.
3. **Hình rơi có thể "lấp" chỗ trống** (như cát) → không còn là "gán bằng base + s"; mô hình khác.
4. **Đếm số "chồng" (thành phần liên thông)** sau mỗi lần rơi → DSU trên các hình giao nhau.
5. **Segment Tree động** (không nén, toạ độ tới `10^9`) → tạo nút khi cần; dùng khi các truy vấn đến trực tuyến (không biết trước toạ độ).

Ví dụ biến thể 2 (hình chữ nhật):

In [22]:
def falling_rectangles(rects):
    """rects[i] = [left, width, height]"""
    tops, ans, best = [], [], 0
    for i, (l, w, h) in enumerate(rects):
        base = 0
        for j in range(i):
            lj, wj, _ = rects[j]
            if l < lj + wj and lj < l + w:
                base = max(base, tops[j])
        tops.append(base + h)
        best = max(best, base + h)
        ans.append(best)
    return ans

print(falling_rectangles([[1, 4, 1], [2, 1, 5], [0, 10, 2]]))   # [1, 6, 8]
assert falling_rectangles([[l, s, s] for l, s in ex1]) == falling_cells(ex1)

[1, 6, 8]


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Từng ô | Mảng độ cao từng đơn vị | `O(n · S)` | `O(max toạ độ)` | Trọng tài khi test, toạ độ nhỏ |
| 2. Từng cặp | `top[i] = s_i + max top[j]` của các hình giao | `O(n²)` | `O(n)` | `n ≤ ~3000`; dễ viết nhất |
| 3. Danh sách khoảng | Hàm bậc thang: cắt tại `l, r`, lấy max, gộp | `O(n²)` tệ nhất, khấu hao tốt | `O(n)` | Thao tác "gán đoạn", code ngắn |
| 4. Segment Tree | Nén toạ độ + max/gán đoạn lazy | `O(n log n)` | `O(n)` | `n` lớn, kỹ thuật tổng quát |

## 17. Bài tập

**Bài 1 (đoán output).** `fallingSquares([[1,1],[2,1],[1,2]])`, `([[3,2],[1,2],[2,2]])`, `([[1,3],[4,3],[2,3]])`?

**Bài 2 (tính tay).** Lập bảng như 8.1 cho `[[2,2],[1,4],[5,1],[5,2]]`.

**Bài 3 (điền TODO).** Hoàn thiện `overlaps(a, b)` cho hai hình `[l, s]` (khoảng nửa mở).

**Bài 4 (tự viết lại).** Viết lại `my_falling(positions)` bằng danh sách khoảng (Cách 3).

**Bài 5 (mini challenge).** Viết `height_at(positions, queries)`: sau khi mọi hình rơi xong, trả về độ cao mặt đất tại từng điểm `x` trong `queries` (độ cao của ô `[x, x+1)`).

In [23]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def overlaps(a, b):
    # TODO: (l1, s1), (l2, s2) = a, b; trả về l1 < l2 + s2 and l2 < l1 + s1
    pass

# Bài 4 — TODO
def my_falling(positions):
    # TODO: xs=[0], hs=[0]; split(x); i=split(l), j=split(l+s); base=max(hs[i:j]); gộp
    pass

# Bài 5 — TODO
def height_at(positions, queries):
    # TODO: xây hàm bậc thang (Cách 3) rồi với mỗi x: hs[bisect_right(xs, x) - 1]
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: vẽ bằng `draw`.
- Mức 2: `[[1,1],[2,1],[1,2]]`: hình 3 `[1,3)` đè lên cả hai hình cao 1.
- Mức 3: `[1, 1, 3]`; `[2, 2, 4]`; `[3, 3, 6]`.

**Bài 2.**
- Mức 1: hình 2 `[1,5)` giao hình 1 `[2,4)`.
- Mức 2: hình 3 `[5,6)` không giao hình 2 (chạm tại 5).
- Mức 3: đỉnh lần lượt 2, 6, 1, 3 → đáp án `[2, 6, 6, 6]`.

**Bài 3.**
- Mức 1: đổi `[l, s]` thành `[l, l + s)`.
- Mức 2: dùng `<` (nghiêm ngặt).
- Mức 3: `l1 < l2 + s2 and l2 < l1 + s1`.

**Bài 4.**
- Mức 1: `bisect_right(xs, x) - 1`.
- Mức 2: chèn mốc mới với độ cao của khoảng chứa nó.
- Mức 3: `xs[i:j] = [l]; hs[i:j] = [base + s]`.

**Bài 5.**
- Mức 1: dùng lại cách cắt/gộp của Bài 4.
- Mức 2: độ cao tại `x` là `hs` của khoảng chứa `x`.
- Mức 3: đối chiếu với mảng `H` của `falling_cells` (viết lại để trả về `H`).

## 19. Đáp án tham khảo

In [24]:
S = Solution()
# Bài 1
assert S.fallingSquares([[1, 1], [2, 1], [1, 2]]) == [1, 1, 3]
assert S.fallingSquares([[3, 2], [1, 2], [2, 2]]) == [2, 2, 4]
assert S.fallingSquares([[1, 3], [4, 3], [2, 3]]) == [3, 3, 6]

# Bài 2
assert S.fallingSquares([[2, 2], [1, 4], [5, 1], [5, 2]]) == [2, 6, 6, 6]

# Bài 3
def overlaps_ans(a, b):
    (l1, s1), (l2, s2) = a, b
    return l1 < l2 + s2 and l2 < l1 + s1

assert overlaps_ans([1, 2], [2, 3]) and not overlaps_ans([100, 100], [200, 100]) and overlaps_ans([1, 10], [3, 1])
print("Bài 1–3 OK")

Bài 1–3 OK


In [25]:
# Bài 4
def build_steps(positions):
    xs, hs = [0], [0]
    def split(x):
        i = bisect_right(xs, x) - 1
        if xs[i] != x:
            xs.insert(i + 1, x)
            hs.insert(i + 1, hs[i])
            i += 1
        return i
    ans, best = [], 0
    for l, s in positions:
        i = split(l)
        j = split(l + s)
        base = max(hs[i:j])
        xs[i:j] = [l]
        hs[i:j] = [base + s]
        best = max(best, base + s)
        ans.append(best)
    return ans, xs, hs

def my_falling_ans(positions):
    return build_steps(positions)[0]

# Bài 5
def height_at_ans(positions, queries):
    _, xs, hs = build_steps(positions)
    return [hs[bisect_right(xs, x) - 1] for x in queries]

def cells_heights(positions, width):
    H = [0] * (width + 1)
    for l, s in positions:
        base = max(H[l:l + s])
        for x in range(l, l + s):
            H[x] = base + s
    return H

random.seed(31)
for _ in range(1000):
    pos = [[random.randint(1, 10), random.randint(1, 4)] for _ in range(random.randint(1, 7))]
    assert my_falling_ans(pos) == falling_cells(pos)
    H = cells_heights(pos, 16)
    assert height_at_ans(pos, list(range(16))) == H[:16]
print("Bài 4–5 OK (đối chiếu mô phỏng từng ô)")

Bài 4–5 OK (đối chiếu mô phỏng từng ô)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template so từng cặp (`n` nhỏ):**

```python
for i, (l, s) in enumerate(P):
    base = max((top[j] for j < i if l < lj + sj and lj < l + s), default=0)
    top[i] = base + s; best = max(best, top[i]); ans.append(best)
```

**Template "hàm bậc thang" (gán đoạn):**

```python
i = split(l); j = split(r); base = max(hs[i:j]); xs[i:j] = [l]; hs[i:j] = [base + s]
```

**Template Segment Tree:** nén `{l, l+s}`; ô `idx[l] .. idx[l+s] - 1`; `base = query`; `assign(base + s)`; lazy = gán.

**Checklist trước khi submit:**

- [ ] Khoảng **nửa mở** `[l, l+s)`: giao khi `l1 < r2 and l2 < r1` (dấu `<`).
- [ ] `base` = **max** đỉnh các hình bên dưới, không phải hình gần nhất.
- [ ] Đáp án là **max cộng dồn**, không phải đỉnh hình hiện tại.
- [ ] Nén toạ độ: dùng cả `l` và `l + s`; phủ ô tới `idx[l+s] - 1`.
- [ ] Segment Tree: `push` trước khi đi xuống; cập nhật `mx` của nút sau khi gán con.
- [ ] Test `[[1,2],[2,3],[6,1]] → [2,5,5]`, `[[100,100],[200,100]] → [100,100]`.
- [ ] Tên method đúng: `fallingSquares(self, positions)`.